# Load SAHARA (African NLP benchmark) → Delta

SAHARA (UBC-NLP, ACL 2025) — the most extensive African-language LLM benchmark (517 languages, 18 tasks). The raw results are private, but the public Gradio leaderboard exposes them via named API endpoints, so we pull keylessly over HTTP.

Writes four tables to `fso_market_intelligence.frontier_labs`:
- `sahara_scores` — atomic: model × task × language × score
- `sahara_task_scores` — model × task (the Space's per-task score)  ← task type records
- `sahara_language_scores` — model × language (the Space's Language Score)  ← per-language proficiency
- `sahara_leaderboard` — model overall + per-cluster (derived)  ← model leaderboard

Idempotent — full overwrite each run (SAHARA is a static published benchmark).

In [ ]:
%pip install lxml -q

In [ ]:
CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"
BASE    = "https://ubc-nlp-sahara.hf.space"

# task id -> cluster (stable; from the Space's helper.py)
CLUSTERS = {
    "Text Classification": ["xlni", "lid", "news", "sentiment", "topic"],
    "Text Generation":     ["mt_eng2xx", "mt_fra2xx", "mt_xx2xx", "paraphrase", "summary", "title"],
    "MCCR":                ["mmlu", "mgsm", "belebele", "squad_qa"],
    "Token-Level":         ["ner", "phrase", "pos"],
}
TASK_CLUSTER = {t: c for c, ts in CLUSTERS.items() for t in ts}

In [ ]:
import json, urllib.request, io, re, time
import pandas as pd

def _post(path, payload):
    req = urllib.request.Request(BASE + path, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"}, method="POST")
    return json.loads(urllib.request.urlopen(req, timeout=60).read().decode())

def _get(path):
    req = urllib.request.Request(BASE + path, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=120).read().decode("utf-8", "replace")

def call(api, data, retries=3):
    """Call a gradio named endpoint; return the (title, html_table) output list."""
    for a in range(retries):
        try:
            eid = _post(f"/gradio_api/call/{api}", {"data": data})["event_id"]
            stream = _get(f"/gradio_api/call/{api}/{eid}")
            lines = [l[6:] for l in stream.splitlines() if l.startswith("data: ")]
            return json.loads(lines[-1])
        except Exception as e:
            if a == retries - 1: raise
            time.sleep(3 * (a + 1))

def dropdown_choices(label):
    cfg = json.loads(_get("/config"))
    for c in cfg["components"]:
        if c.get("type") == "dropdown" and c.get("props", {}).get("label") == label:
            return [ch[1] if isinstance(ch, list) else ch for ch in c["props"]["choices"]]
    return []

def clean_model(s):
    return re.sub(r"^[^A-Za-z0-9]+", "", str(s)).strip()   # strip leading 🏆/🥈/🥉 etc.

def parse_table(html):
    return pd.read_html(io.StringIO(html))[0]

TASKS = dropdown_choices("Select Task")       # e.g. "Sentiment Analysis (sentiment)"
LANGS = dropdown_choices("Select Language")
print(f"{len(TASKS)} tasks · {len(LANGS)} languages")

## Pull per-task tables → atomic scores + task scores

In [ ]:
atomic_rows, task_rows = [], []
for t in TASKS:
    tid = re.search(r"\(([^)]+)\)\s*$", t)
    tid = tid.group(1) if tid else t
    cluster = TASK_CLUSTER.get(tid, "Other")
    _, html = call("update_task_table", [t])
    df = parse_table(html)
    df["model"] = df["model"].map(clean_model)
    lang_cols = [c for c in df.columns if c not in ("model", "Task Score")]
    for _, r in df.iterrows():
        if "Task Score" in df.columns and pd.notna(r.get("Task Score")):
            task_rows.append({"model": r["model"], "task": tid, "cluster": cluster,
                              "task_score": float(r["Task Score"])})
        for lc in lang_cols:
            v = r[lc]
            if pd.notna(v):
                atomic_rows.append({"model": r["model"], "task": tid, "cluster": cluster,
                                    "language": str(lc), "score": float(v)})
    print(f"  {tid:12} {len(df)} models × {len(lang_cols)} langs")
print(f"atomic rows: {len(atomic_rows)} · task rows: {len(task_rows)}")

## Pull per-language tables → language proficiency

In [ ]:
lang_rows = []
for lang in LANGS:
    _, html = call("update_lang_table", [lang])
    df = parse_table(html)
    if "Language Score" not in df.columns:
        continue
    df["model"] = df["model"].map(clean_model)
    for _, r in df.iterrows():
        if pd.notna(r.get("Language Score")):
            lang_rows.append({"model": r["model"], "language": lang, "language_score": float(r["Language Score"])})
print(f"language rows: {len(lang_rows)}")

## Write Delta tables

In [ ]:
from pyspark.sql import functions as F

def write(rows, name):
    df = spark.createDataFrame(rows).withColumn("captured_at", F.current_date())
    (df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{name}"))
    print(f"{name}: {df.count()} rows")
    return df

write(atomic_rows, "sahara_scores")
task_df = write(task_rows, "sahara_task_scores")
write(lang_rows, "sahara_language_scores")

# leaderboard: overall (mean task score) + per-cluster mean, one row per model
overall = task_df.groupBy("model").agg(F.round(F.avg("task_score"), 2).alias("overall_score"),
                                       F.count("task").alias("tasks_evaluated"))
by_cluster = (task_df.groupBy("model").pivot("cluster").agg(F.round(F.avg("task_score"), 2)))
leaderboard = (overall.join(by_cluster, "model", "left")
               .withColumn("captured_at", F.current_date())
               .orderBy(F.col("overall_score").desc_nulls_last()))
(leaderboard.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.sahara_leaderboard"))
print("sahara_leaderboard:", leaderboard.count(), "models")
display(leaderboard.limit(10))